# 01 — Register queryable raw Bronze external tables

Register `lakehouse_demo.bronze.customers`, `orders`, `deliveries`, and `events`
over the **existing initial entity directories**. No source copies, managed-table
writes, cleanup, type normalization, deduplication, or quarantine are performed.
Run `00_setup_catalog` first. Install the same base project wheel or source checkout.

## Configuration

This is the only environment configuration section. Copy the values used in notebook 00.
The verified namespace is `frgpslcqn7sm`; tenancy `oraemeaaidata2` is not the namespace.
Do not supply passwords, API keys, tokens, wallets, or local OCI config files here.

In [ ]:
# CONFIGURATION: edit environment values only here, in both notebooks.
# These are non-secret values verified in docs/object-storage-execution.md.
CONFIG = {
    "catalog": "lakehouse_demo",
    "namespace": "frgpslcqn7sm",  # Object Storage namespace, NOT oraemeaaidata2
    "bucket": "CristinaVaras-bucket-AIDP",
    "prefix": "demos/lakehouse-to-ai/source-data/v1/seed-42/reference-2026-10-28",
    "reference_date": "2026-10-28",  # Fixed synthetic-data reference date
    "seed": 42,
    "batch_id": "initial",         # Do not include incremental in this milestone
}
# Optional: absolute driver-visible checkout path when using source imports.
# Leave None when the project wheel is installed on the AIDP compute cluster.
REPO_ROOT = None


In [ ]:
from pathlib import Path
import sys

if REPO_ROOT is not None:
    source_dir = Path(REPO_ROOT) / "src"
    if not (source_dir / "aidp_lakehouse" / "aidp_bronze.py").is_file():
        raise RuntimeError("REPO_ROOT must identify a driver-visible checkout with src/aidp_lakehouse")
    sys.path.insert(0, str(source_dir))

from aidp_lakehouse.aidp_bronze import BronzeConfig

cfg = BronzeConfig(**CONFIG)
if "spark" not in globals():
    raise RuntimeError("Attach authenticated AIDP compute and run with its supplied spark session")
print("Spark runtime:", spark.version)
print("Configured source root (not an ingestion wildcard):", cfg.root_uri)


## Source and platform preflight

AIDP does not permit external tables beneath volumes or on bucket/directory locations
already claimed by another external table/volume. Check Master Catalog for root/ancestor
claims, including other demos. These four initial directories are disjoint, but global
catalog/volume ownership cannot be verified from this local artifact. Stop on a location
conflict; do not move/copy files or create a different layout as a fallback.

CSV columns (including numeric-looking fields and dates) are explicit STRINGs. JSONL
uses one object per line; `payload` is a string-to-string map, timestamps are STRINGs,
and `row_version` is BIGINT as in the JSON source. `_corrupt_record` preserves parser
failures. PERMISSIVE mode retains malformed rows; no DROPMALFORMED or date casts are used.
The initial batch is clean; deliberate invalid dates are in the unregistered incremental
batch and remain unchanged there. The raw string contract also retains them if a later
milestone explicitly registers that batch.

In [ ]:
import json
from aidp_lakehouse.aidp_bronze import (
    ENTITIES, table_ddl, read_control_documents, register_bronze,
    verify_bronze, provenance_sample_sql,
)

# Exact control object reads, not recursive business/RAG ingestion or registered tables.
manifest, ingestion_metadata = read_control_documents(spark, cfg)
print("Manifest source:", cfg.control_uri("manifest.json"))
print("Ingestion metadata source:", cfg.control_uri("ingestion_metadata.json"))
print("Initial expected rows:", json.dumps(manifest["row_counts"], sort_keys=True))
print("Batch ID:", manifest["batch_id"], "Extracted at:", manifest["extracted_at"])
for entity in ENTITIES:
    print("Table:", cfg.table(entity))
    print("External LOCATION:", cfg.location(entity))
    print("Only expected source object:", cfg.file_uri(entity))
    print(table_ddl(cfg, entity) + ";")


## Register missing tables, preserve existing definitions

Existing target tables are checked before creating absent ones. Type, provider, location,
raw schema and critical parser options must agree. A conflict stops execution; no ALTER,
DROP, REPLACE, data-write or location rewrite is used. Registration is metadata-only.

In [ ]:
registration = register_bronze(spark, cfg)
print(json.dumps(registration, indent=2))


## Verify registered definitions and locations

The generated statements above show the intended definitions. DESCRIBE below shows
actual catalog definitions, including location, provider and reader storage properties.

In [ ]:
spark.sql(f"SHOW TABLES IN {cfg.catalog}.bronze").show(truncate=False)
for entity in ENTITIES:
    print("Registered definition:", cfg.table(entity))
    spark.sql(f"DESCRIBE TABLE EXTENDED {cfg.table(entity)}").show(100, truncate=False)


## Compare raw counts with the uploaded initial manifest

Expected values for this fixed source run are 600 customers, 30,600 orders,
30,000 deliveries and 94,572 events. The assertion uses the actual uploaded manifest,
not those explanatory constants. Each table must read exactly its one initial source file.
No filter or deduplication is applied to counts or samples. Timestamp ranges below are
lexical ranges over unchanged ISO-8601 source strings, not date conversions.

In [ ]:
verification = verify_bronze(spark, cfg, manifest)
print(json.dumps(verification, indent=2, default=str))


## Small raw samples and provenance

Source files carry `source_created_at`, `source_updated_at` and `row_version`,
but no batch_id column. Batch ID is derived explicitly from the initial path/control
metadata. Extraction time comes from the control manifest; it is not an actual Bronze
ingestion timestamp. The provenance columns below exist only in these SELECT results;
no source or external-table schema is altered to add them.

In [ ]:
for entity in ENTITIES:
    print("Raw sample:", cfg.table(entity))
    spark.sql(f"SELECT * FROM {cfg.table(entity)} LIMIT 5").show(truncate=False)
    print("Available source timestamps and derived batch provenance:")
    spark.sql(provenance_sample_sql(cfg, entity, manifest["extracted_at"])).show(truncate=False)
print("Bronze runtime verification completed in this attached AIDP session.")


## Official references and execution status

DDL follows [Oracle AIDP SQL Grammar](https://docs.oracle.com/en/cloud/paas/ai-data-platform/aidug/sql-grammar.html),
[external-table support and limitations](https://docs.oracle.com/en/cloud/paas/ai-data-platform/aidug/tables.html),
and Oracle's [standard-catalog notebook sample](https://github.com/oracle-samples/oracle-aidp-samples/blob/main/data-engineering/transformation/liquid-clustering/financial_services_delta_liquid_clustering_demo.ipynb).
See `docs/aidp-bronze-setup.md` for imports, permissions and source contracts.

**Runtime verification: PENDING.** No cells in this checked-in notebook have been executed against AIDP.
No local OCI key or SQLcl Autonomous connection is used by this notebook.